In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import pytz

dbutils.widgets.removeAll()
# Parámetros del Job
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "", "Fecha Proceso (DDMMYYYY)")

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

# Configuración Hora Perú
peru_tz = pytz.timezone('America/Lima')
ts_peru = datetime.now(peru_tz).strftime('%Y-%m-%d %H:%M:%S')

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

catalog = f"{dbutils.widgets.get('env')}_fraud"

df_bronze = spark.table(f"{catalog}.bronze.catalogo_mcc_bronze")

df_silver = df_bronze.select(
    col("codigo_mcc").cast("int"),
    initcap(col("descripcion")).alias("descripcion"),
    when(col("descripcion").contains("Retail"), "Ventas Minoristas")
        .when(col("descripcion").contains("Travel"), "Viajes")
        .when(col("descripcion").contains("Food"), "Alimentación")
        .otherwise("Otros").alias("categoria_grupo"),
    current_timestamp().alias("ingestion_timestamp"),
    lit("99999999").alias("process_date") # Catálogo es maestro fijo
).dropDuplicates(["codigo_mcc"])

target = f"{catalog}.silver.mcc_silver"
dt = DeltaTable.forName(spark, target)
dt.alias("t").merge(df_silver.alias("s"), "t.codigo_mcc = s.codigo_mcc") \
  .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_final_silver = spark.table(target)
total_registros_silver = df_final_silver.count()
registros_hoy = df_final_silver.filter(col("process_date") == process_date).count()

print("═" * 70)
print(f" REPORTE DE TABLA FÍSICA: {target.upper()} ".center(70, "═"))
print("═" * 70)
print(f"{'Catálogo.Esquema.Tabla':<30} : {target}")
print(f"{'Total registros acumulados':<30} : {total_registros_silver:,}")
print(f"{'Registros cargados/actualizados hoy':<30} : {registros_hoy:,}")
print(f"{'Última sincronización (Perú)':<30} : {ts_peru}")
print("═" * 70)

print(f"\n🔍 VISTA PREVIA DE LA TABLA SILVER (Top 3 recientes):")

display(
    df_final_silver
    .orderBy(col("ingestion_timestamp").desc())
    .limit(3)
)